# Análise de clientes - Bronze Layer

Análise exploratória da tabela `bronze.clientes`, feita antes do tratamento na Silver. O objetivo é identificar os formatos de data em `data_cadastro` e os valores não numéricos em `idade`. O notebook usa apenas a tabela da Bronze.

In [0]:
from pyspark.sql import functions as F

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"

df_bronze_clientes = spark.table(f"{bronze_schema}.clientes")

## Funções auxiliares

`mostrar_padrao_data` troca cada dígito de uma coluna de data por 9 e conta quantas vezes cada padrão aparece. Assim, "25/10/2023" vira "99/99/9999" e "2023-05-12" vira "9999-99-99", o que permite ver de uma vez todos os formatos usados.

In [0]:
def mostrar_padrao_data(df, datetime_column):
    # Substitui qualquer dígito numérico (0-9) pelo número 9
    # Assim, "25/10/2023" vira "99/99/9999" e "2023-05-12" vira "9999-99-99"
    df_padroes = df.withColumn(
        "padrao_data",
        F.regexp_replace(F.col(datetime_column), r"\d", "9")
    )

    # Agrupa e conta a frequência de cada padrão encontrado no seu dataset
    df_padroes.groupBy("padrao_data").count().orderBy(F.desc("count")).show()

## 1. Padrões de data em data_cadastro

A primeira consulta mostra as datas escritas com `-`. A segunda conta todos os padrões encontrados na coluna.

In [0]:
%sql

select * from vcredit_medallion.bronze.clientes where data_cadastro like '%-%'

In [0]:
# Verifica padrões de data na coluna data_cadastro
mostrar_padrao_data(df_bronze_clientes, "data_cadastro")

# Padrões identificados: dd/MM/yyyy e yyyy-MM-dd

## 2. idade com caracteres não numéricos

Exibe as linhas em que `idade` não é formada apenas por dígitos.

In [0]:
%sql

-- verificar se a idade não é formada apenas por dígitos
select * from vcredit_medallion.bronze.clientes where idade NOT REGEXP '^[0-9]+$';



## Conclusões e tratamentos previstos para a Silver

| Problema encontrado | Evidência | Tratamento |
|---|---|---|
| `data_cadastro` em dois formatos | Padrões `dd/MM/yyyy` e `yyyy-MM-dd` (seção 1) | Silver: converter para data testando os dois formatos |
| `idade` com caracteres não numéricos | Resultado da seção 2 | Silver: manter só os dígitos, converter para inteiro e anular idades fora de 18 a 120 anos |